# पाठ १३ - एजेन्ट स्मृति


## सेटअप

यो नोटबुकले **Microsoft Agent Framework** (MAF) प्रयोग गरेर **स्थायी स्मरणशक्ति** भएको ट्राभल बुकिंग एजेन्ट कसरी बनाउने देखाउँछ।

तपाईंले एजेन्ट स्मरणशक्तिका विभिन्न प्रकारहरू — कार्यशील, छोटो अवधि, र लामो अवधि — ले कसरी एजेन्टले कुराकानीहरूमा जानकारीलाई कायम राख्ने र प्रयोग गर्ने तरिका सिक्नु हुनेछ।

**पूर्वशर्तहरू:**
- एक Azure AI Foundry परियोजना जसमा डिप्लोय गरिएको च्याट मोडल (जस्तै `gpt-4o-mini`) छ।
- Azure CLI मा लगइन गरिएको — आफ्नो टर्मिनलमा `az login` चलाउनुहोस्।
- `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Azure AI Foundry परियोजना अन्तर्को।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंले डिप्लोय गरेको मोडलको नाम।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजेण्ट स्मृतिका प्रकारहरू

एआई एजेण्टहरूले विभिन्न प्रकारका स्मृतिहरू प्रयोग गर्नसक्छन्, जसले हरेकले फरक उद्देश्य पूरा गर्छ:

### कार्य स्मृति
संवाद थ्रेड आफैं — एक सत्रमा आदानप्रदान गरिएका सन्देशहरू। एजेण्टले समन्वय कायम राख्नको लागि सोही थ्रेडमा पहिलेका सन्देशहरूलाई फेरि हेर्न सक्छ। MAF मा यो **`agent.create_session()`** मार्फत सिर्जना गरिन्छ, जुन `AgentSession` फर्काउँछ।

### छोटो अवधि स्मृति
कार्य वा सत्रको अवधिभर टिकिरहने जानकारी तर स्थायी रूपमा संग्रहित नभएका। उदाहरणका लागि, एजेण्ट बहु-चरण नियोजन संवादको समयमा तथ्यहरू जम्मा गर्न सक्छ र अन्तिम यात्रा तालिका उत्पादन गर्न तिनीहरू प्रयोग गर्न सक्छ।

### दीर्घकालीन स्मृति
पसन्द र तथ्यहरू जुन **सत्रहरू पार गरेर** टिकिरहन्छन्। पुन: आउने प्रयोगकर्ताले आफ्नो आहार प्रतिबन्ध वा यात्रा शैली दोहोर्याउन नपर्ने। दीर्घकालीन स्मृति सामान्यतया बाह्य भण्डारणद्वारा समर्थित हुन्छ — डेटाबेस, फाइल, वा भेक्टर सूचकाङ्क — र उपकरणहरू मार्फत एजेण्टमा प्रस्तुत गरिन्छ।


## कार्यशील स्मृति सत्रहरूसँग

सबैभन्दा सरल स्मृतिको रूप हो कुरा सत्र। जब तपाईं उही सत्र (जुन `agent.create_session()` बाट सिर्जना गरिएको हो) निरन्तर `agent.run()` कलहरूमा पास गर्नुहुन्छ, एजेन्टले त्यो कुराकानीको पूरै इतिहास देख्छ र पहिलेका विवरणहरू सम्झना गर्न सक्दछ।

आउनुहोस् एक यात्रा एजेन्ट बनाऔं र कार्यशील स्मृतिको प्रदर्शन गरौं।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजेन्टले बजेटलाई ठीकसँग सम्झायो किनभने दुबै सन्देशकर्मीले एउटै सत्र साझा गरेका छन्। यो हो **कार्यशील स्मरण** — यो केवल सत्रको आयुका लागि मात्र हुन्छ।

### नयाँ थ्रेडसँग के हुन्छ?

यदि हामीले **नयाँ** सत्र सिर्जना गर्छौं भने, एजेन्टलाई अघिल्लो कुराकानीको कुनै स्मरण हुँदैन:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## लामो समयसम्म सम्झने ढाँचा

प्रयोगकर्ता प्राथमिकताहरू **सत्रहरूमा पार गर्दै** सम्झनको लागि, हामीसँग यस्तो स्थायी संग्रह हुनुपर्छ जुन कुराकानी धागो बाहिर जीवन्त हुन्छ। एजेन्टले यो संग्रहलाई **उपकरणहरू** मार्फत पहुँच गर्दछ — त्यस्ता कार्यहरू जुन यसले सूचना सुरक्षित गर्न र पुनः प्राप्त गर्न कल गर्न सक्छ।

तल्लोमा हामीले एक साधारण स्मृति प्राथमिकता भण्डार कार्यान्वयन गरिरहेका छौं (उत्पादनमा तपाईंले यसलाई डेटाबेस वा भेक्टर सूचीको साथ समर्थन गर्नुहुनेछ) र यसलाई उपकरणहरूका रूपमा खुलासा गरिरहेका छौं जुन एजेन्टले प्रयोग गर्न सक्छ।

### वास्तुकला
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### परिदृश्य १ — पहिलो पल्ट प्रयोगकर्ताले वार्षिकोत्सव यात्रा बुक गर्छन्

सारा पहिलो पटक भ्रमण गर्न आउँछिन्। एजेन्टले उनका प्राथमिकताहरू उपकरणहरू मार्फत संग्रह गर्नुपर्छ र होटलहरू सिफारिस गर्न तिनीहरूलाई प्रयोग गर्नुपर्छ।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### अवस्था २ — सारा हप्ताहरु पछि फर्किन्छिन्

सारा एउटा **पुर्ण रूपमा नयाँ थ्रेड** सुरु गर्छिन् (नयाँ सत्रको अनुकरण गर्दै)। काम गर्ने स्मृति खाली छ, तर दीर्घकालीन प्राथमिकता भण्डारमा अझै पनि उनको जानकारी छ। एजेन्टले यो पुनःप्राप्त गर्नुपर्छ र सिफारिसहरू व्यक्तिगत बनाउन प्रयोग गर्नुपर्छ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

यो पाठमा तपाईँले तीन प्रकारका एजेन्ट मेमोरी र तिनीहरूलाई Microsoft Agent Framework सँग कसरी कार्यान्वयन गर्ने भनेर सिक्नुभयो:

| मेमोरी प्रकार | MAF संयन्त्र | आयु अवधि |
|---|---|---|
| **काम गर्ने** | `agent.create_session()` | एकल संवाद |
| **छोटो-समय** | थ्रेड भित्र संचय गरिएको सन्दर्भ | एकल कार्य / सत्र |
| **दीर्घकालीन** | बाह्य संग्रह जसलाई `@tool` फङ्क्सनहरूले पहुँच गर्छन् | सत्रहरू बीचमा |

### मुख्य बुँदाहरू
1. **`agent.create_session()`** काम गर्ने मेमोरी प्रदान गर्छ — एजेन्टले सत्र भित्रको पूर्ण संवाद इतिहास देख्छ।
2. **नयाँ सत्रहरूले सन्दर्भ गुमाउँछन्** — दीर्घकालीन मेमोरी बिना एजेन्टले विगतका संवादहरू सम्झन सक्दैन।
3. **`@tool` फङ्क्सनहरूले पुलको काम गर्छन्** — तीले एजेन्टलाई स्थायी सञ्चयबाट जानकारी बचत र पुन:प्राप्त गर्न दिन्छन्।
4. **व्यक्तिकरण समयसँग सुधार हुन्छ** — जति धेरै प्राथमिकताहरू सङ्ग्रह गरिन्छ, एजेन्टको सिफारिसहरू उति नै राम्रो हुन्छन्।

### वास्तविक-विश्व अनुप्रयोगहरू
- **ग्राहक सेवा**: ग्राहकको इतिहास र प्राथमिकताहरू सम्झनुहोस्
- **व्यक्तिगत सहायकहरू**: दिन वा हप्ताहरुमा सन्दर्भ कायम राख्नुहोस्
- **स्वास्थ्य सेवा**: रोगीको जानकारी र प्राथमिकताहरू ट्र्याक गर्नुहोस्
- **ई-वाणिज्य**: इतिहासको आधारमा व्यक्तिगत किनमेल

### अर्को चरणहरू
- इन-मेमोरी डिक्टलाई डेटाबेस वा भेक्टर स्टोर (जस्तै Azure AI Search) सँग प्रतिस्थापन गर्नुहोस्
- समय-संवेदी जानकारीका लागि मेमोरी समाप्ति थप्नुहोस्
- साझा मेमोरीसहित बहु-एजेन्ट प्रणालीहरू बनाउनुहोस्
- ज्ञान-ग्राफसमेत्तित मेमोरीका लागि Cognee नोटबुक अन्वेषण गर्नुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**स्पष्टिकरण**:  
यस दस्तावेजलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) को प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताको प्रयास गर्छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा गलतिहरु हुनसक्छन्। मूल दस्तावेज यसको मूल भाषामा मात्र विश्वसनीय स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीको लागि व्यावसायिक मानव अनुवाद सिफारिश गरिन्छ। यस अनुवादको प्रयोगबाट आउने कुनै पनि गलतफहमी वा गलत व्याख्याको लागि हामी जिम्मेवार हुनुहुँदैन।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
